In [18]:
import os
import sys
import warnings
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings(
    "ignore",
    message="Could not find the number of physical cores.*",
    category=UserWarning,
)

import joblib
import numpy as np
import pandas as pd
from sqlalchemy import text

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_access.connection import engine


pd.set_option("display.max_columns", None)
print("Корень проекта:", PROJECT_ROOT)
print("Версия scikit-learn:")
import sklearn
print(sklearn.__version__)

Корень проекта: D:\PROJECTS\average_task_performance
Версия scikit-learn:
1.9.0


In [19]:
query_path = PROJECT_ROOT / "data_access" / "queries" / "training_dataset.sql"
query = text(query_path.read_text(encoding="utf-8"))

issue_df = pd.read_sql_query(query, engine)

print("Количество строк:", len(issue_df))
issue_df.head()

Количество строк: 251920


,ID,Project_ID,Creation_Date,Description_Issue_Length,Duration_Days,initial_priority,initial_type,Description_Project_Length
0,85,1,2016-02-24 10:51:02,249,303.1658,Minor,Bug,146
1,86,1,2016-02-23 18:31:38,125,303.8457,Minor,Improvement,146
2,89,1,2016-02-22 15:27:31,384,0.1347,Minor,Story,146
3,90,1,2016-02-17 14:28:31,18,3.4999,Minor,Bug,146
4,91,1,2016-02-16 14:59:06,579,3.2120,Minor,Improvement,146


## Хронологическое разделение

Модель обучается на первых 70% задач. Следующие 15% используются как validation, последние 15% — как test.

In [20]:
issue_df = issue_df.sort_values(
    ["Creation_Date", "ID"]
).reset_index(drop=True)

train_end = int(len(issue_df) * 0.70)
validation_end = int(len(issue_df) * 0.85)

train_df = issue_df.iloc[:train_end].copy()
validation_df = issue_df.iloc[train_end:validation_end].copy()
test_df = issue_df.iloc[validation_end:].copy()

pd.DataFrame(
    {
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "date_min": [
            train_df["Creation_Date"].min(),
            validation_df["Creation_Date"].min(),
            test_df["Creation_Date"].min(),
        ],
        "date_max": [
            train_df["Creation_Date"].max(),
            validation_df["Creation_Date"].max(),
            test_df["Creation_Date"].max(),
        ],
    },
    index=["train", "validation", "test"],
)

,rows,date_min,date_max
train,176344,2003-04-08 00:48:51,2017-04-24 18:28:43
validation,37788,2017-04-24 18:38:00,2018-11-08 14:59:22
test,37788,2018-11-08 15:01:59,2020-10-22 15:16:24


## Формирование `X` и `y`

In [21]:
feature_columns = [
    "Description_Issue_Length",
    "Description_Project_Length",
    "initial_type",
    "initial_priority"
]
target_column = "Duration_Days"

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

pd.DataFrame(
    {
        "X_shape": [X_train.shape, X_validation.shape, X_test.shape],
        "y_shape": [y_train.shape, y_validation.shape, y_test.shape],
    },
    index=["train", "validation", "test"],
)

,X_shape,y_shape
train,"(176344, 4)","(176344,)"
validation,"(37788, 4)","(37788,)"
test,"(37788, 4)","(37788,)"


## Preprocessing

`Description_Length` передаётся как число. `OrdinalEncoder` превращает строковые тип и приоритет в числовые категории. Неизвестная категория при запросе получает `-1`.

In [22]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            ["Description_Issue_Length", "Description_Project_Length"],
        ),
        (
            "categories",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1,
            ),
            ["initial_type", "initial_priority"],
        ),
    ],
    verbose_feature_names_out=False,
)

preprocessing

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categories', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per trans

In [23]:
regressor = HistGradientBoostingRegressor(
    categorical_features=[False, False, True, True],
    learning_rate=0.08,
    max_iter=250,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42,
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("regressor", regressor),
    ]
)

model = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

model

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",False
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as

In [24]:
started_at = perf_counter()
model.fit(X_train, y_train)
training_seconds = perf_counter() - started_at

print(f"Обучение завершено за {training_seconds:.2f} сек.")

Обучение завершено за 1.93 сек.


In [25]:
def calculate_metrics(y_true, prediction):
    prediction = np.maximum(prediction, 0.0)
    return {
        "MAE_days": mean_absolute_error(y_true, prediction),
        "Median_AE_days": median_absolute_error(y_true, prediction),
        "RMSE_days": root_mean_squared_error(y_true, prediction),
        "R2": r2_score(y_true, prediction),
    }


metrics = {}

for split_name, features, target in (
    ("train", X_train, y_train),
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
):
    prediction = model.predict(features)
    metrics[split_name] = calculate_metrics(target, prediction)

metrics_df = pd.DataFrame(metrics).T
metrics_df

,MAE_days,Median_AE_days,RMSE_days,R2
train,47.066984,12.964295,88.285598,-0.155701
validation,44.550182,14.423876,83.696775,-0.136494
test,37.571890,14.874034,68.927443,-0.101404


## Сохранение модели для Backend

In [26]:
model_path = PROJECT_ROOT / "artifacts" / "duration_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)

model_bundle = {
    "model": model,
    "feature_columns": feature_columns,
    "target_column": target_column,
    "trained_at": datetime.now(UTC).isoformat(),
    "training_rows": len(X_train),
    "training_seconds": training_seconds,
    "metrics": metrics,
}

joblib.dump(model_bundle, model_path)
print("Модель сохранена:", model_path)

Модель сохранена: D:\PROJECTS\average_task_performance\artifacts\duration_model.joblib
